# Using PRQL to query data



- [Query language: wikipedia]( https://en.wikipedia.org/wiki/Query_language )
- [PRQL - Pipelined Relational Query Language, pronounced “Prequel”
]( https://prql-lang.org/ )
- [PRQL in DuckDB]( https://duckdb.org/community_extensions/extensions/prql )


In [ ]:
%%capture pip_output
%%bash
pip install pyprql


In [ ]:
import duckdb
import prqlc
import pandas as pd
import pyprql.pandas_accessor


In [ ]:
data = {
    'region': ['West'] * 20,
    'state': ['NM']*10 + ['AZ']*10,
    'county': ['Bernalillo']*5 + ['Santa Fe']*5 + ['Maricopa']*5 + ['Pima']*5,
    'city': [
        'Albuquerque', 'Albuquerque', 'Rio Rancho', 'Tijeras', 'Cedar Crest',
        'Santa Fe', 'Santa Fe', 'Espanola', 'Edgewood', 'Glorieta',
        'Phoenix', 'Mesa', 'Chandler', 'Scottsdale', 'Tempe',
        'Tucson', 'Marana', 'Oro Valley', 'Sahuarita', 'South Tucson'
    ],
    'tract': [f'T-{i:03d}' for i in range(1, 21)],
    'block': [f'B-{i:04d}' for i in range(101, 121)],
    'population': [2500, 1800, 3100, 450, 600, 2100, 1900, 800, 1200, 300,
                   5000, 4200, 3800, 3100, 2900, 4500, 1100, 950, 700, 400]
}
# data


In [ ]:
df = pd.DataFrame(data)
df


## Using Pipelined Relational Query Language ( PRQL )




### Example 0


In [ ]:
prql_query = '''
  from df
'''


In [ ]:
sql_query = prqlc.compile(prql_query)
print(sql_query)


In [ ]:
duckdb.sql(sql_query)


### Example 1


In [ ]:
# Using pandas
df[['city', 'county', 'population']]


In [ ]:
# using PRQL
query = '''
  select { city, county, population }
'''

df.prql.query( query )


In [ ]:
# Using PRQL and DuckDB
sql_query = prqlc.compile('''
  from df
  select { city, county, population }
''')
duckdb.sql(sql_query)


### Example 2



In [ ]:
# Using pandas
df[['city', 'county', 'population']].head(3)


In [ ]:
# using PRQL
query = '''
  select { city, county, population }
  take 3
'''

df.prql.query( query )



In [ ]:
# Using PRQL and DuckDB
sql_query = prqlc.compile('''
  from df
  select { city, county, population }
  take 3
''')
duckdb.sql(sql_query)


### Example 3



In [ ]:
# using pandas
(
df
  [['city', 'county', 'population']]
  [ df['population'] > 2000 ]
  .sort_values('population', ascending=False)
  .head(3)
)

In [ ]:
# using PRQL
query = '''
  select { city, county, population }
  filter ( population > 2000 )
  sort { -population }
  take 3
'''

df.prql.query( query )


In [ ]:
# Using PRQL and DuckDB
sql_query = prqlc.compile('''
  from df
  select { city, county, population }
  filter ( population > 2000 )
  sort { -population }
  take 3
''')
duckdb.sql(sql_query)


In [ ]:
# Using PRQL and DuckDB -- reordered
sql_query = prqlc.compile('''
  from df                               # the data frame
  filter ( population > 2000 )          # filter the rows
  select { city, county, population }   # select the columns
  sort { -population }                  # sort, descending
  take 3                                # filter the first 3
''')
print(sql_query)
duckdb.sql(sql_query)
